# 00. Setup and checks

Run this notebook once before training. It checks the GPU, secrets, data and model, then measures the real speed to replace the estimates in the plan.

**Requirements:** enable a GPU and add the `HF_TOKEN` and `WANDB_API_KEY` secrets as described in the README.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Change this to your repo URL. For a private repo, add a GITHUB_TOKEN secret on Colab.
REPO_URL = "https://github.com/YOUR-USERNAME/vihsd-lora.git"

def github_token():
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

def find_repo():
    places = [Path.cwd(), *Path.cwd().parents, Path("/content/vihsd-lora"), Path("/kaggle/working/vihsd-lora")]
    places += list(Path("/kaggle/input").glob("*")) + list(Path("/kaggle/input").glob("*/vihsd-lora"))
    return next((place for place in places if (place / "src" / "vihsd_lora").is_dir()), None)

repo = find_repo()
if repo is None:
    if "YOUR-USERNAME" in REPO_URL:
        raise ValueError("Set REPO_URL to your GitHub repo URL, then run this cell again.")
    repo = Path("/content" if Path("/content").exists() else "/kaggle/working") / "vihsd-lora"
    token = github_token()
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url, str(repo)], check=True)
elif (repo / ".git").exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=False)
if not os.environ.get("VIHSD_SKIP_INSTALL"):
    # Colab ships an old torchao that makes peft fail when attaching LoRA. This repo does not use torchao, so remove it.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
sys.path.insert(0, str(repo / "src"))
print("Repo:", repo)

In [ ]:
from vihsd_lora.config import Settings
from vihsd_lora.env import default_output_root, describe_gpu, detect_platform, free_space_gb, load_secrets

print("Platform:", detect_platform())
print("GPU:", describe_gpu())
print("Secrets found:", load_secrets())

settings = Settings.from_env(output_root=default_output_root())
free_gb = free_space_gb(settings.output_root)
print("Results are saved to:", settings.output_root)
print(f"Free space: {free_gb} GB")
if free_gb < 6:
    print("Warning: less than 6GB left. A full fine-tune run needs about 3GB, so clean up Drive and empty the trash.")

## Check the data

The cell below stops if the per-label counts do not match the reference ViHSD version.

In [ ]:
import pandas as pd
from vihsd_lora.data import label_counts, length_report, load_vihsd
from vihsd_lora.metrics import majority_baseline

dataset = load_vihsd(settings)
display(pd.DataFrame(label_counts(dataset)).T)

baseline = majority_baseline(dataset["test"]["labels"])
print(f"Always predicting CLEAN on test: Acc {baseline['accuracy']:.2f}, macro F1 {baseline['macro_f1']:.2f}")

## Check the tokenizer and comment lengths

`truncated_pct` is the share of comments longer than `max_length` that get truncated.

In [ ]:
from vihsd_lora.modeling import load_tokenizer

tokenizer = load_tokenizer(settings.model_name)
print(length_report(dataset["train"], tokenizer, settings.max_length))

## Check the parameter counts

Expected with ViSoBERT: LoRA r=8 trains 887,811 parameters, full fine-tune trains 97,568,259 parameters.

In [ ]:
from vihsd_lora.config import RUNS
from vihsd_lora.modeling import build_model, count_parameters

rows = []
for key, run in RUNS.items():
    if key == "E":
        continue
    rows.append({"run": key, "description": run.describe(), **count_parameters(build_model(run, settings))})
display(pd.DataFrame(rows))

## Measure the real speed

Runs 200 steps for LoRA and for full fine-tune. The final table is the estimated training time based on the measured speed, excluding dev evaluation.

In [ ]:
from vihsd_lora.speedtest import estimate_hours, measure

SPEED_STEPS = 200
speed_lora = measure("A", steps=SPEED_STEPS, settings=settings)
speed_full = measure("B", steps=SPEED_STEPS, settings=settings)
print(speed_lora)
print(speed_full)
display(estimate_hours({"lora": speed_lora["seconds_per_step"], "full": speed_full["seconds_per_step"]}, settings))